# MLP Classifier - Predict Student Dropout & Academic Success
**Dataset:** UCI - Predict Students' Dropout and Academic Success  
**Model:** Multilayer Perceptron (Neural Network)

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('data.csv', sep=';')
print(f"Shape: {df.shape}")
print(f"\nTarget Distribution:\n{df['Target'].value_counts()}")
print(f"\nMissing Values: {df.isnull().sum().sum()}")
df.head()

## 3. Preprocessing

### 3a. Encode Target Variable

In [ ]:
le = LabelEncoder()
df['Target_encoded'] = le.fit_transform(df['Target'])
print(f"Encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

### 3b. Separate Features and Target

In [ ]:
X = df.drop(columns=['Target', 'Target_encoded'])
y = df['Target_encoded']
X.columns = [col.strip() for col in X.columns]
print(f"Features: {X.shape[1]} | Samples: {X.shape[0]}")

### 3c. Feature Scaling (StandardScaler)
Neural networks are sensitive to feature magnitude.  
StandardScaler transforms each feature to mean=0, std=1.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Scaling applied: mean=0, std=1")

### 3d. Train-Test Split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## 4. Build and Train MLP

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),   # 2 hidden layers
    activation='relu',               # ReLU activation
    solver='adam',                   # Adam optimizer
    max_iter=500,                    # max epochs
    random_state=42,
    early_stopping=True,             # prevent overfitting
    validation_fraction=0.1
)

mlp.fit(X_train, y_train)
print("Training complete.")
print(f"Iterations: {mlp.n_iter_}")

## 5. Predictions and Accuracy

In [ ]:
y_pred = mlp.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"MLP Accuracy: {accuracy*100:.2f}%")

## 6. Classification Report

In [ ]:
print(classification_report(y_test, y_pred, target_names=le.classes_))

## 7. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'MLP Confusion Matrix (Accuracy: {accuracy*100:.2f}%)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## 8. Training Loss Curve

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(mlp.loss_curve_, color='steelblue', linewidth=2)
plt.title('MLP Training Loss Curve')
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Observations
- **Accuracy:** 75.37%
- **Graduate** class has the best recall (89%) — easiest to predict
- **Enrolled** class has the lowest recall (41%) — it's a transitional state that overlaps with both Dropout and Graduate
- **Dropout** detection is decent (74% recall) — useful for early intervention
- The loss curve shows smooth convergence, confirming the model trained properly